# Silver -- `stg_orders`

Cleansed order headers, one row per order

**Source:** `bronze_commerce_orders`  
**Business key:** `order_id`  
**Load pattern:** `full_refresh`

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_bronze, writes to lh_silver. Both must be
# attached to this notebook; lh_silver must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "lh_silver"
source_item = "lh_bronze"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="stg_orders",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="silver", table_name="stg_orders")
print(f"load_id={load_id}  environment={environment}  table=stg_orders")


In [ ]:
# ---- Read bronze -------------------------------------------------
# Depends on stg_customers, stg_order_items being built first -- the spec's
# depends_on establishes this ordering.
df = spark.read.table(f"{source_item}.bronze_commerce_orders")
rows_in = df.count()
dq.record_input(rows_in)
print(f"read {rows_in:,} rows from bronze_commerce_orders")


In [ ]:
# ---- Column mapping ----------------------------------------------
# Lifted verbatim from mappings/silver.yaml so the notebook is
# self-contained and auditable without opening the spec.
columns = [   {   'source': 'order_id',
        'target': 'order_id',
        'type': 'string',
        'nullable': False,
        'key': 'business'},
    {   'source': 'customer_id',
        'target': 'customer_id',
        'type': 'string',
        'nullable': False,
        'key': 'foreign'},
    {   'source': 'order_date',
        'target': 'order_date',
        'type': 'timestamp',
        'nullable': False},
    {   'source': None,
        'target': 'order_date_key',
        'type': 'date',
        'nullable': False,
        'expression': 'to_date(order_date)',
        'description': 'Join key to dim_date'},
    {   'source': 'total_price',
        'target': 'order_total',
        'type': 'decimal',
        'nullable': False,
        'precision': 18,
        'scale': 2},
    {   'source': 'total_price',
        'target': 'order_total_as_reported',
        'type': 'decimal',
        'nullable': False,
        'precision': 18,
        'scale': 2},
    {   'source': None,
        'target': 'order_total_was_corrected',
        'type': 'boolean',
        'nullable': False,
        'expression': 'abs(order_total_as_reported - order_total) > 0.01'},
    {   'source': 'status',
        'target': 'status',
        'type': 'string',
        'nullable': False,
        'transforms': ['trim', 'lower'],
        'allowed_values': ['pending', 'confirmed', 'shipped', 'delivered', 'cancelled'],
        'on_violation': 'quarantine'},
    {   'source': None,
        'target': 'is_revenue',
        'type': 'boolean',
        'nullable': False,
        'expression': "status in ('confirmed', 'shipped', 'delivered')"},
    {   'source': 'items_count',
        'target': 'items_count_as_reported',
        'type': 'integer',
        'nullable': False},
    {   'source': None,
        'target': 'items_count',
        'type': 'integer',
        'nullable': False,
        'expression': 'count(order_items where order_id = this.order_id)'},
    {   'source': 'created_at',
        'target': 'created_at',
        'type': 'timestamp',
        'nullable': False},
    {   'source': 'updated_at',
        'target': 'updated_at',
        'type': 'timestamp',
        'nullable': False}]


In [ ]:
# ---- Cleansing rules ---------------------------------------------
# Each rule returns kept and rejected rows. Rejected rows accumulate
# into the quarantine frame so nothing is lost without a reason.
quarantine = None

def apply(result):
    """Collect rejects and carry the kept frame forward."""
    global quarantine, df
    if result.rejected is not None and not result.rejected.isEmpty():
        quarantine = (result.rejected if quarantine is None
                      else quarantine.unionByName(result.rejected,
                                                  allowMissingColumns=True))
    if result.corrected_count:
        dq.record_corrected(result.corrected_count)
    df = result.kept
    return result


In [ ]:
# cast_types
result = apply(get_rule("cast_types")(df, ctx, columns=columns))
dq.record_rule("cast_types", result)


In [ ]:
# apply_transforms
result = apply(get_rule("apply_transforms")(df, ctx, columns=columns))
dq.record_rule("apply_transforms", result)


In [ ]:
# deduplicate  (handles DUP-002; on_reject: quarantine)
result = apply(get_rule("deduplicate")(df, ctx, keys=['order_id'], keep='latest', order_by=['updated_at desc', 'created_at desc']))
dq.record_rule("deduplicate", result)


In [ ]:
# drop_null_required  (on_reject: quarantine)
#   An order with no date cannot be placed on a timeline, so it cannot
#   contribute to any time-based measure. Quarantined for upstream follow-up
#   rather than defaulted to a date it did not happen on.
result = apply(get_rule("drop_null_required")(df, ctx, columns=['order_date']))
dq.record_rule("drop_null_required", result)


In [ ]:
# quarantine_negative_total  (handles SIGN-001; on_reject: quarantine)
#   A negative header total is a refund recorded in the wrong place.
#   Treating it as revenue would understate the true figure, and flipping
#   its sign would invent a sale. Quarantined until upstream issues a proper
#   credit note.
result = apply(get_rule("quarantine_negative_total")(df, ctx, column='order_total', min_exclusive=0))
dq.record_rule("quarantine_negative_total", result)


In [ ]:
# enforce_allowed_values  (on_reject: quarantine)
result = apply(get_rule("enforce_allowed_values")(df, ctx, column='status', values=['pending', 'confirmed', 'shipped', 'delivered', 'cancelled']))
dq.record_rule("enforce_allowed_values", result)


In [ ]:
# enforce_referential_integrity  (handles REF-001; on_reject: quarantine)
#   Orders whose customer was purged upstream. Quarantined rather than
#   attached to an "Unknown Customer" placeholder, which would hide a real
#   upstream deletion problem behind a plausible-looking dimension member.
result = apply(get_rule("enforce_referential_integrity")(df, ctx, column='customer_id', references='stg_customers.customer_id'))
dq.record_rule("enforce_referential_integrity", result)


In [ ]:
# recompute_total_from_lines  (handles RECON-002; on_reject: correct_and_flag)
#   The lines are the source of truth; the header is a cached rollup.
#   Recomputed from the already-corrected lines, which is why this rule runs
#   after stg_order_items is built.
result = apply(get_rule("recompute_total_from_lines")(df, ctx, target='order_total', from_table='stg_order_items', join_on='order_id', aggregate='sum(subtotal)', tolerance=0.01, keep_original_as='order_total_as_reported'))
dq.record_rule("recompute_total_from_lines", result)


In [ ]:
# recompute_items_count  (on_reject: correct_and_flag)
result = apply(get_rule("recompute_items_count")(df, ctx, target='items_count', from_table='stg_order_items', join_on='order_id', aggregate='count(*)', keep_original_as='items_count_as_reported'))
dq.record_rule("recompute_items_count", result)


In [ ]:
# add_record_hash
result = apply(get_rule("add_record_hash")(df, ctx, exclude=['_processed_at', '_load_id']))
dq.record_rule("add_record_hash", result)


In [ ]:
# ---- Write -------------------------------------------------------
final_columns = [c["target"] for c in columns]
out = df.select(*[c for c in final_columns if c in df.columns])

# Audit columns from 00-platform.yaml `audit_columns.silver`.
out = (out
    .withColumn("_processed_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id)))

rows_out = out.count()
out.write.mode("overwrite").format("delta").saveAsTable("stg_orders")
dq.record_output(rows_out)
print(f"wrote {rows_out:,} rows to stg_orders")

if quarantine is not None:
    rejected_count = quarantine.count()
    # Overwrite, matching silver's own write mode. Silver is fully
    # rebuilt each run, so an appended quarantine would accumulate
    # rejects from previous builds and break the layer-level identity
    # count(bronze) == count(silver) + count(quarantine).
    (quarantine.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta").saveAsTable("stg_orders_quarantine"))
    dq.record_quarantined(rejected_count)
    print(f"quarantined {rejected_count:,} rows to stg_orders_quarantine")
else:
    rejected_count = 0


In [ ]:
# ---- Reconciliation ----------------------------------------------
# SILVER-RECON-003: every input row is accounted for. A shortfall
# means a rule dropped rows without quarantining them, which is a
# framework bug rather than a data problem.
accounted = rows_out + rejected_count
if accounted != rows_in:
    raise AssertionError(
        f"row loss: {rows_in:,} in, {rows_out:,} out, "
        f"{rejected_count:,} quarantined, {rows_in - accounted:,} unaccounted"
    )
print(f"reconciled: {rows_in:,} = {rows_out:,} kept + {rejected_count:,} quarantined")

dq.flush()
